In [ ]:
from google.colab import drive
import os
import pandas as pd
import re
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
drive.mount('/content/drive') #Run this line to mount your drive to colab environment. Needs to be done with each session
# misa_path = 'content/drive/Sharedwithme/MISA/' #This line should be run by everyone that isn't Calvin
misa_path = '/content/drive/MyDrive/get/Fine_tune_MISA/DSD_reruns' #This line is to be run by Calvin only
student_folders = os.listdir(misa_path)
print("Student folders in MISA path:", student_folders)


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
from google.colab import drive
import os
import pandas as pd
import re
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
drive.mount('/content/drive') #Run this line to mount your drive to colab environment. Needs to be done with each session
# misa_path = 'content/drive/Sharedwithme/MISA/' #This line should be run by everyone that isn't Calvin
misa_path = '/content/drive/MyDrive/get/Fine_tune_MISA/DSD_reruns' #This line is to be run by Calvin only
student_folders = os.listdir(misa_path)
print("Student folders in MISA path:", student_folders)
# https://drive.google.com/drive/folders/1NhAWdaJC6jGnckMao2Z3noKlqli_RDLb?usp=sharing #This is the filepath to the MISA folder with data from the output files

all_experiments = {}
for student_name in student_folders:
  file_path = os.path.join(misa_path, student_name)
  filenames = os.listdir(file_path)
  student_experiments = []
  for files in filenames:
    filename_path = os.path.join(file_path, files)
    epochs, losses, misis = [], [], []
    with open(filename_path, 'r') as file:
      for line in file:
        match = re.search(r'epoch: (\d+)\s+loss: ([\d.]+)\s+MISI: ([\d.]+)', line)
        if match:
          epoch = int(match.group(1))
          loss = float(match.group(2))
          misi = float(match.group(3))
          epochs.append(epoch)
          losses.append(loss)
          misis.append(misi)
      student_experiments.append(pd.DataFrame({
        'epoch': epochs,
        'loss': losses,
        'misi': misis,
        'log_loss': np.log10(losses),
        'log_misi': np.log10(misis)}))
    print(f"Contents of {files} successfully read")
  print(f'Student folder /{student_name} successfully read')
  all_experiments[f'{student_name}'] = student_experiments
# print(all_experiments)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/get/Fine_tune_MISA/DSD_reruns'

In [ ]:
total_max_loss = -float('inf')
total_min_loss = float('inf')
total_max_misi = -float('inf')
total_min_misi = float('inf')
for student in all_experiments:
  for experiment in all_experiments[student]:
    max_loss = experiment['loss'].max()
    min_loss = experiment['loss'].min()
    print(f"  DataFrame: Max loss = {max_loss}, Min loss = {min_loss}")
    range = max_loss - min_loss
    max_misi = experiment['log_misi'].max()
    min_misi = experiment['log_misi'].min()
    print(f"  DataFrame: Max misi = {max_misi}, Min misi = {min_misi}")
    if max_loss > total_max_loss:
      total_max_loss = max_loss
    if min_loss < total_min_loss:
      total_min_loss = min_loss
    if max_misi > total_max_misi:
      total_max_misi = max_misi
    if min_misi < total_min_misi:
      total_min_misi = min_misi

for student in all_experiments:
  file_path = os.path.join(misa_path, student)
  filenames = os.listdir(file_path)
  count = 1
  for ix, experiment in enumerate(all_experiments[student]):
    # sns.relplot(
    # data=experiment,
    # x="epoch", y="loss",
    # hue="misi").fig.suptitle(f'{student} Experiment {count}', fontsize=14)  # Workaround for single title
    # sns.relplot(
    # data=experiment,
    # x="epoch", y="misi",
    # hue="loss").fig.suptitle(f'{student} Experiment {count}', fontsize=14)  # Workaround for single title

    # sns.scatterplot(x=experiment['epoch'], y=experiment['loss'], color='blue', label='Loss Values')
    # sns.scatterplot(x=experiment['epoch'], y=experiment['misi'], color='red', label='MISI Values')
    # plt.xlabel('Epochs')
    # plt.ylabel('MISI Values')
    # plt.title(f'{student} Experiment {count}')
    # plt.show()

    fig, ax1 = plt.subplots(1, 1, figsize=(20,5))
    # ax1.scatter(experiment['epoch'], np.log10(experiment['loss'] - experiment['loss'].min() + 1) + 1, color='blue', label='Loss Values', s=2)
    ax1.plot(experiment['epoch'], np.log10(experiment['loss'] - experiment['loss'].min() + 1) + 1, color='blue', label='Loss Values')

    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('log_Loss', color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')
    # change axis 1 to log scale:
    ax1.set_yscale('log')
    ax1.set_ylim(None, np.log10(total_max_loss - experiment['loss'].min() + 1) + 1)

    ax2 = ax1.twinx()
    # ax2.scatter(experiment['epoch'], experiment['log_misi'], color='red', label='MISI Values', s=2)
    ax2.plot(experiment['epoch'], experiment['log_misi'], color='red', label='MISI Values')

    ax2.set_ylabel('log_MISI', color='red')
    ax2.tick_params(axis='y', labelcolor='red')
    ax2.set_ylim(total_min_misi, total_max_misi)

    # plt.xlabel('Epochs'
    plt.title(f'{student} Experiment {count} - {filenames[ix]}')
    fig.tight_layout()
    # Adjust axis limits to be from 0 to highest value - minimum for loss.
    plt.show()
    count += 1

  #Grab lowest and highest loss, subtract value when plotting and add 1.
  #Grab lowest and highest misi and scale every plot identically

NameError: name 'all_experiments' is not defined